# Notebook 05: Full Pipeline & Lead Generation

Notebook ini mendemonstrasikan end-to-end pipeline chatbot AI untuk layanan Nasi Kotak Catering.

Pipeline menggabungkan:
1. **Conversation Manager**: State/memory user.
2. **RAG Service**: Retrieval konteks produk dan knowledge base.
3. **LLM Service (Groq)**: Prompting, intent recognition, entity extraction dengan schema JSON.
4. **Sales Engine**: Logika rekomendasi, penentuan `purchase_intent`, upsell, dan cross-sell.
5. **Lead Manager**: Trigger penangkapan prospek (lead) ketika intent memuncak dan pembuatan CTA link WhatsApp.

Semua orkestrasi di atas dibungkus dalam `src.pipeline.ChatPipeline`.

## 5.1 Setup & Inisialisasi

In [1]:
import os
import sys
from dotenv import load_dotenv

load_dotenv()
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.database import init_db, SessionLocal, Lead, Base, engine
from src.pipeline import ChatPipeline
from src.config import PROJECT_ROOT

# Reset database (opsional) agar tidak menumpuk saat dijalankan ulang
Base.metadata.drop_all(bind=engine)

# Inisialisasi Database (SQLite)
init_db()
db = SessionLocal()

# Inisialisasi Full Pipeline
pipeline = ChatPipeline()
print("✅ Pipeline siap.")

Database initialized successfully.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Berhasil memuat index FAISS dengan 22 vektor.
✅ Pipeline siap.


--- 
## 5.2 Helper Function untuk Menampilkan Hasil Chat

In [2]:
import textwrap

def print_chat_result(user_msg, result):
    print("=" * 80)
    print(textwrap.fill(f"User: {user_msg}", 80))
    print("-" * 80)
    
    if "error" in result:
        print(f"❌ Error: {result.get('error')}")
        return
        
    print(textwrap.fill(f"Bot: {result.get('reply', '')}", 80))
    print("-" * 80)
    print(f"📌 Intent: {result.get('intent', 'N/A')}")
    print(f"🔥 Purchase Intent: {result.get('purchase_intent', 'N/A')}")
    
    entities = result.get('entities', {})
    if any(v is not None for v in entities.values()):
        print(f"📦 Extracted Entities:")
        for k, v in entities.items():
            if v is not None:
                print(f"   - {k}: {v}")
                
    if result.get('actions'):
        print(f"⚙️ Actions: {result.get('actions')}")
        
    if result.get('needs_handover'):
        print(f"⚠️ Handover Triggered: {result.get('handover_reason')}")
        
    if result.get('lead_status') == 'captured':
        print(f"✅ LEAD CAPTURED! (ID: {result.get('lead_id')})")
        if result.get('whatsapp_link'):
            print(f"🔗 WhatsApp CTA: {result.get('whatsapp_link')}")
    print("=" * 80)
    print()

--- 
## 5.3 Scenario A: Customer Individu (Arisan, Budget 25rb)
Skenario ini akan menunjukkan bagaimana `purchase_intent` memuncak dan Lead di-capture.

In [3]:
session_a = "demo_session_A"
scenario_a_messages = [
    "Halo, saya mau cari nasi kotak untuk arisan bulan depan.",
    "Budgetnya sekitar 25 ribu per box.",
    "Butuh 50 box untuk tanggal 15 Oktober.",
    "Oke saya mau pesan paket Ayam Kampung itu."
]

for msg in scenario_a_messages:
    result = pipeline.chat(msg, session_id=session_a, db=db)
    print_chat_result(msg, result)

User: Halo, saya mau cari nasi kotak untuk arisan bulan depan.
--------------------------------------------------------------------------------
Bot: Hai kak, untuk arisan bulan depan, berapa jumlah box yang dibutuhkan? Ada
budget per box? Dan kapan serta di mana acara?
--------------------------------------------------------------------------------
📌 Intent: product_inquiry
🔥 Purchase Intent: LOW
📦 Extracted Entities:
   - session_id: demo_session_A
   - event_type: arisan
   - purchase_intent: LOW
   - messages: [{'sender': 'user', 'text': 'Halo, saya mau cari nasi kotak untuk arisan bulan depan.'}, {'sender': 'bot', 'text': 'Hai kak, untuk arisan bulan depan, berapa jumlah box yang dibutuhkan? Ada budget per box? Dan kapan serta di mana acara?'}]
⚙️ Actions: ['ask_quantity', 'ask_budget', 'ask_date', 'ask_location']

User: Budgetnya sekitar 25 ribu per box.
--------------------------------------------------------------------------------
Bot: Kak, untuk arisan dengan budget sekitar Rp

--- 
## 5.4 Scenario B: Corporate (Meeting 100 Box, Web Redirect)
Skenario ketika intent = ordering, bot akan direct ke URL website.

In [4]:
session_b = "demo_session_B"
scenario_b_messages = [
    "Butuh katering meeting kantor 100 orang, budget 30rb, apa yang cocok?",
    "Paket Bebek Mantap sepertinya pas. Saya ambil itu.",
    "Kirim ke Sudirman hari Jumat jam 12, tolong diproses sekarang."
]

for msg in scenario_b_messages:
    result = pipeline.chat(msg, session_id=session_b, db=db)
    print_chat_result(msg, result)

[INFO] TPM limit tercapai. Menunggu 56.8s...
User: Butuh katering meeting kantor 100 orang, budget 30rb, apa yang cocok?
--------------------------------------------------------------------------------
Bot: Kak, untuk meeting kantor 100 orang dengan budget 30k per box, paket yang
paling cocok adalah Nasi Kotak Bebek/Gurami seharga Rp 27.000 per box. Dengan
pesanan 100 box, kamu juga dapat menikmati promo gratis 1 box + ongkir.   Kak,
kalau sudah setuju, silakan kunjungi website kami untuk memesan. 😊
--------------------------------------------------------------------------------
📌 Intent: recommendation
🔥 Purchase Intent: HIGH
📦 Extracted Entities:
   - session_id: demo_session_B
   - quantity: 100
   - budget_per_box: 30000.0
   - event_type: meeting kantor
   - purchase_intent: HIGH
   - messages: [{'sender': 'user', 'text': 'Butuh katering meeting kantor 100 orang, budget 30rb, apa yang cocok?'}, {'sender': 'bot', 'text': 'Kak, untuk meeting kantor 100 orang dengan budget 30k per bo

--- 
## 5.5 Scenario C: Event Organizer (Custom Menu / Handoff)
Skenario ketika user memiliki kebutuhan dietary yang rumit dan memaksa fallback safety-net `needs_handover`.

In [5]:
session_c = "demo_session_C"
scenario_c_messages = [
    "Halo, saya EO. Mau pesan 80 box untuk besok.",
    "Saya butuh 10 porsi menu khusus vegan strict, bisa dijamin bebas kontaminasi produk hewani dari dapur?"
]

for msg in scenario_c_messages:
    result = pipeline.chat(msg, session_id=session_c, db=db)
    print_chat_result(msg, result)

User: Halo, saya EO. Mau pesan 80 box untuk besok.
--------------------------------------------------------------------------------
Bot: Kak, terima kasih sudah memilih Nasi Kotak Pak D. Untuk 80 box, kami
rekomendasikan Minibox (Rp 17.000/box). Apakah acara ini corporate event,
wedding, atau lainnya? Setelah itu, kami bisa konfirmasi harga dan promo.
Silakan kunjungi website kami untuk memesan.  Untuk melanjutkan pemesanan,
silakan melalui halaman web kami ya kak 🛒✨ 👉 https://ayambakarpakd.com/order
--------------------------------------------------------------------------------
📌 Intent: recommendation
🔥 Purchase Intent: READY_TO_ORDER
📦 Extracted Entities:
   - session_id: demo_session_C
   - quantity: 80
   - purchase_intent: READY_TO_ORDER
   - messages: [{'sender': 'user', 'text': 'Halo, saya EO. Mau pesan 80 box untuk besok.'}, {'sender': 'bot', 'text': 'Kak, terima kasih sudah memilih Nasi Kotak Pak D. Untuk 80 box, kami rekomendasikan Minibox (Rp 17.000/box). Apakah acara ini 

--- 
## 5.6 Scenario D: Anti-Hallucination
Skenario ketika user menanyakan produk yang jelas tidak ada di knowledge base (misal: sushi).

In [6]:
session_d = "demo_session_D"
result = pipeline.chat("Halo, ada menu sushi box nggak?", session_id=session_d, db=db)
print_chat_result("Halo, ada menu sushi box nggak?", result)

User: Halo, ada menu sushi box nggak?
--------------------------------------------------------------------------------
Bot: Maaf kak, kami tidak menyediakan menu sushi box. Silakan hubungi admin kami
untuk informasi lebih lanjut.  Untuk hal ini, saya hubungkan ke admin kami ya
kak 🙏 Admin 3 - ⁠Shinta: https://wa.me/081914176108
--------------------------------------------------------------------------------
📌 Intent: product_inquiry
🔥 Purchase Intent: LOW
📦 Extracted Entities:
   - session_id: demo_session_D
   - purchase_intent: LOW
   - messages: [{'sender': 'user', 'text': 'Halo, ada menu sushi box nggak?'}, {'sender': 'bot', 'text': 'Maaf kak, kami tidak menyediakan menu sushi box. Silakan hubungi admin kami untuk informasi lebih lanjut.\n\nUntuk hal ini, saya hubungkan ke admin kami ya kak 🙏\nAdmin 3 - \u2060Shinta: https://wa.me/081914176108'}]
⚙️ Actions: ['handover_admin']
⚠️ Handover Triggered: Menu sushi box tidak tersedia dalam layanan kami.



--- 
## 5.7 Lead Report
Mari kita query SQLite database untuk melihat Lead yang berhasil di-capture selama simulasi di atas.

In [7]:
import pandas as pd

leads = pipeline.lead_manager.get_all_leads(db)
lead_data = []
for l in leads:
    lead_data.append({
        "ID": l.id,
        "Event": l.event_type,
        "Qty": l.quantity,
        "Budget": l.budget,
        "Location": l.location,
        "Purchase Intent": l.purchase_intent,
        "Created": l.created_at
    })

df_leads = pd.DataFrame(lead_data)
df_leads

,ID,Event,Qty,Budget,Location,Purchase Intent,Created
0,7,NaN,80,NaN,NaN,HIGH,2026-08-14 08:22:26.783509
1,6,NaN,80,NaN,NaN,READY_TO_ORDER,2026-08-14 08:21:28.792167
2,5,meeting kantor,100,30000.0,Sudirman,READY_TO_ORDER,2026-08-14 08:21:27.024523
3,4,meeting kantor,100,30000.0,NaN,READY_TO_ORDER,2026-08-14 08:21:25.904469
4,3,meeting kantor,100,30000.0,NaN,HIGH,2026-08-14 08:21:25.040265
5,2,arisan,50,25000.0,NaN,READY_TO_ORDER,2026-08-14 08:20:26.246196
6,1,arisan,50,25000.0,NaN,HIGH,2026-08-14 08:20:25.073936
